In [ ]:
library(Seurat)
library(dplyr)
library(data.table)
library(ggplot2)
source("~/Projects/heads/clustering.r")

In [ ]:
data_dir = "/gpfs/gibbs/pi/braun/zy325"

In [ ]:
obj = readRDS(file.path(data_dir,"processed","theta1_dims50","scrcc_lymphoid.rds"))

In [ ]:
# Recover AIR assay back to RNA
counts1 = LayerData(obj, assay = "RNA", layer = "counts")
counts2 = LayerData(obj, assay = "AIR", layer = "counts")
counts = rbind(counts1, counts2)
obj[["RNA"]] = CreateAssay5Object(counts = counts)
obj[["AIR"]] = NULL

In [ ]:
obj = clustering(obj, keep_c_genes = T, 
                plot_QC_metrics = FALSE,
                group.by.vars = "batch_lab",
                harmony_theta=1,dims = 1:20)

In [ ]:
#saveRDS(obj,file=file.path(data_dir,"processed","theta1_dims50","scrcc_lymphoid_clustered_addCgenes.rds"))

obj = readRDS(file.path(data_dir,"processed","theta1_dims50","scrcc_lymphoid_clustered_addCgenes.rds"))

In [ ]:
options(repr.plot.width=8,repr.plot.height=10)
VlnPlot(obj,features = c(
    "CD3D","CD3E","CD3G","TRAC","TRBC1","TRBC2","TRDC",
    "CD8A","CD4",
    "NCAM1","FCGR3A",
    "IL7R","TBX21","IFNG",
    "GATA3","PTGDR2","IL1RL1",
    "KIT","RORC","IL23R"
   ),pt.size = 0,stack = TRUE,flip = TRUE)

In [ ]:
options(repr.plot.width=7,repr.plot.height=7)
DimPlot(obj,label=TRUE) + NoLegend()

In [ ]:
### TCR mapping ###

tcrs = list.dirs("/gpfs/gibbs/project/braun/zy325/scrcc/raw/tcr_airrflow_output/cellranger",recursive = F)
tcrs = lapply(file.path(tcrs,"outs","filtered_contig_annotations.csv"),function(x){
    cr = fread(x) %>% mutate(
        sample_id2=gsub("^.*cellranger\\/","",gsub("\\/outs.*$","",x)),
        sample_barcode=paste0(sample_id2,'_',barcode))
    return(cr)
}) %>% rbindlist

trbs = tcrs %>% filter(chain == "TRB") 

obj$barcode = gsub("^.*removed_","",obj$name)
obj$sample_barcode = paste0(obj$sample_id2,"_",obj$barcode)
obj$wTCR = obj$sample_barcode %in% tcrs$sample_barcode
obj$wTRB = obj$sample_barcode %in% trbs$sample_barcode

options(repr.plot.width=15,repr.plot.height=7)
DimPlot(obj,group.by = "wTCR") | DimPlot(obj,group.by = "wTRB")

samples_missingtcr = paste0("SCRCC",c("14NORM","15","77","78"))

obj@meta.data %>% 
    filter(!sample_id2 %in% samples_missingtcr) %>%
    group_by(`RNA_snn_res.0.5`) %>%
    summarize(n=n(),
        n_wTCR=sum(wTCR),prop_wTCR=n_wTCR/n,
        n_wTRB=sum(wTRB),prop_wTRB=n_wTRB/n) %>% arrange(desc(prop_wTCR))

In [ ]:
options(repr.plot.width=16,repr.plot.height=8)

DimPlot(obj,group.by = "batch_lab") | DimPlot(obj,group.by = "batch_seq_rna") #+ NoLegend()

In [ ]:
options(repr.plot.width=10,repr.plot.height=10)
VlnPlot(obj,features = c("nCount_RNA","nFeature_RNA","percent.mt"),pt.size = 0,stack = T,flip = T)

In [ ]:
options(repr.plot.width=15,repr.plot.height=10)
VlnPlot(obj,features = c(
    "CD3D","CD3E","CD3G",
    "TRBC1","TRBC2","TRAC",
    "CD8A","CD8B","CD4","CD69","FOXP3",
    "CD79A","CD79B","MS4A1","MZB1","JCHAIN",
    "NCAM1","NCR1","FCGR3A",
    "MKI67","TOP2A",
    "SPP1","VEGFA",
    "CD68","S100A9","FCN1","C1QC",
    "HBB","PPBP","EPCAM","ALDOB","PECAM1","COL1A1","PTPRC"),pt.size = 0,stack = TRUE,flip = TRUE)

In [ ]:
# Check previous annotation
table(obj$`RNA_snn_res.0.5`[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.5`[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.5`[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing=TRUE) #%>% sum

In [ ]:
# Put C genes back
Tcell = c(0:3,5,8,11,12,15)
B = c(9,13,16)
NK = c(6,10,14)
ILC = c(4)

# Contamination
# Hypoxic: SPP1+ VEGFA+ - 7
# Small clusters - 17

obj$lineage3 = case_when(
    obj$`RNA_snn_res.0.5` %in% Tcell~"T",
    obj$`RNA_snn_res.0.5` %in% B~"B",
    obj$`RNA_snn_res.0.5` %in% NK~"NK",
    obj$`RNA_snn_res.0.5` %in% ILC~"ILC",
    TRUE~"contamination")

In [ ]:
options(repr.plot.width=16,repr.plot.height=7)
DimPlot(obj,group.by = "lineage3") | DimPlot(obj,group.by = "wTCR") #+ NoLegend()

In [ ]:
options(repr.plot.width=8.5,repr.plot.height=7)
DimPlot(obj,cells.highlight = obj$name[obj$anno_cd8t == "CD8Tex_NMF3"])

In [ ]:
saveRDS(subset(obj,lineage3=="ILC"),file="scrcc_lymphoid_c4.rds")

In [ ]:
library(ggsankey)
options(repr.plot.width=7,repr.plot.height=7)

obj@meta.data %>% 
    make_long(lineage3,anno3) %>%
    ggplot(aes(x = x, 
               next_x = next_x, 
               node = node, 
               next_node = next_node,
               fill = factor(node),
               label = node)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3.5, color = 1, fill = "white") +
    theme_sankey(base_size = 16) + NoLegend()

In [ ]:
library(ggsankey)

obj@meta.data %>% 
    filter(!is.na(anno_cd8t)) %>%
    make_long(lineage3,anno_cd8t) %>% # 
    ggplot(aes(x = x, 
               next_x = next_x, 
               node = node, 
               next_node = next_node,
               fill = factor(node),
               label = node)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3, color = 1, fill = "white") +
    theme_sankey(base_size = 16) + NoLegend()

In [ ]:
table(obj$lineage3[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing = TRUE)
table(obj$lineage3[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing = TRUE)
table(obj$lineage3[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing = TRUE)

In [ ]:
obj$lineage3_clusters = obj$`RNA_snn_res.0.5`
obj$`RNA_snn_res.0.5` = NULL
obj$seurat_clusters = NULL

In [ ]:
m = FindMarkers(obj,`ident.1` = c(1),only.pos = TRUE,logfc.threshold = .6)
m %>% filter(p_val_adj<0.05) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)